# Notebook 30 — Latent / RMM Moisture-Convection Phase Diagnostics
**Project:** ENSO-BSISO SSL — MJO moisture-constraint experiment (Zhang et al. 2020)
**Author:** Jiayi (jh9141@nyu.edu)

Measures, **before** any physics loss, what the real MJO does — using the validated own-ERA5 RMM (nb24)
as the primary phase clock (SSL/BT latents as secondary/negative-control). For each field we take the
**first circular harmonic** `A_f(x) = <f(t,x) e^{i theta(t)}>` over active MJO days and report the
**phase offset** vs convection `(-OLR')`:

- `delta_theta(q_col, conv)` ≈ 0  -> moisture-mode-like (Pr ∝ column q, Zhang §5)
- `delta_theta(q_low, conv)` large positive (q leads, quarter cycle) -> skeleton-like recharge (§4.6)
- `delta_theta(dq/dt, conv)` -> the propagation tendency (§5, Fig 9/11)
- **Rossby-Kelvin ratio** (u850 asymmetry, §7 Fig 22) and **BL-convergence lead** (§7) -> trio-interaction

Everything is also **ENSO-stratified** (El Nino / La Nina / Neutral) — the project's headline.

Sign convention: convection proxy `conv = -OLR'`. `theta` is oriented so it increases with eastward MJO
progression (checked against official RMM phase). `delta_theta = arg(A_field) - arg(A_conv)` in degrees,
wrapped to (-180, 180]; positive = field leads convection (located east of convection).

---

## Cell 1 — Load X_MJO, own-RMM, labels, processed moisture

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR     = f'{PROJECT_DIR}/MJO'
PROC        = f'{MJO_DIR}/data/processed'
MOIST       = f'{MJO_DIR}/moisture_constraints/data/processed'
OUT         = f'{MJO_DIR}/moisture_constraints/results/diagnostics'
os.makedirs(OUT, exist_ok=True)

X      = np.load(f'{PROC}/X_MJO.npy')                  # (N,3,1,180) ch0 u850', ch1 OLR', ch2 u200'
lons   = np.load(f'{PROC}/longitudes_mjo.npy')
labels = pd.read_csv(f'{PROC}/labels_aligned_mjo.csv', parse_dates=['date'])
pcs    = np.load(f'{PROC}/mjo_rmm_own_pcs.npy')        # (N,2) BoM-aligned own RMM
qcol   = np.load(f'{MOIST}/qcol_mjo_processed.npy')    # (N,180); NaN where moisture missing
qlow   = np.load(f'{MOIST}/qlow_mjo_processed.npy')
_divp  = f'{MOIST}/divlow_mjo_processed.npy'
divl   = np.load(_divp) if os.path.exists(_divp) else None   # OPTIONAL (BL-convergence)
if divl is None: print('NOTE: divlow not found -> BL-convergence cell will be skipped.')

u850 = X[:,0,0,:]; olr = X[:,1,0,:]; conv = -olr        # convection proxy
conv_x = -olr
phase  = labels['phase'].values.astype(int)
amp    = labels['amplitude'].values
enso   = labels['enso_category'].values
weak   = labels['weak_mjo'].values.astype(bool)
N = len(labels)
assert pcs.shape[0]==N==qcol.shape[0], 'row misalignment between X/own-RMM/moisture'

# restrict to active MJO days that ALSO have moisture coverage (handles partial download)
has_moist = np.isfinite(qcol).all(axis=1) & np.isfinite(qlow).all(axis=1)
active = (~weak) & (amp >= 1.0) & has_moist
n_all_active = int(((~weak) & (amp >= 1.0)).sum())
print(f'N days {N}  active(all) {n_all_active}  active&moisture-covered {int(active.sum())}'
      f'  ({100*has_moist.mean():.0f}% of days have moisture)')
if has_moist.mean() < 0.99:
    cov_dates = pd.DatetimeIndex(labels['date'])[has_moist]
    print(f'  PARTIAL moisture: covered {cov_dates.min().date()}..{cov_dates.max().date()};'
          f' ENSO-stratified bins will be smaller. Complete nb28/nb29 for the full record.')

## Cell 2 — Latent phase + orientation check vs official RMM phase

In [ ]:
theta_own = np.arctan2(pcs[:,1], pcs[:,0])           # radians (-pi,pi]

# orient so increasing theta == increasing official phase (eastward). Use circular correlation sign.
ph_ang = (phase-1)/8.0*2*np.pi                        # official phase 1..8 -> angle
def circ_corr(a, b):
    a=a-np.angle(np.mean(np.exp(1j*a))); b=b-np.angle(np.mean(np.exp(1j*b)))
    return float(np.sum(np.sin(a)*np.sin(b))/np.sqrt(np.sum(np.sin(a)**2)*np.sum(np.sin(b)**2)))
m=active & (phase>=1)&(phase<=8)
cc = circ_corr(theta_own[m], ph_ang[m])
flipped=False
if cc < 0:
    theta_own = -theta_own; cc = -cc; flipped=True
print(f'circular corr(theta_own, official phase) = {cc:.3f}  flipped={flipped}')

fig,ax=plt.subplots(1,2,figsize=(12,4.5))
ax[0].scatter(np.degrees(theta_own[m]), phase[m]+np.random.uniform(-.3,.3,m.sum()), s=3, alpha=.2)
ax[0].set_xlabel('theta_own (deg)'); ax[0].set_ylabel('official RMM phase'); ax[0].set_title(f'orientation (circ corr={cc:.2f})')
# mean official phase per theta bin
b=np.linspace(-np.pi,np.pi,17); bc=.5*(b[1:]+b[:-1])
mp=[phase[m & (theta_own>=b[i]) & (theta_own<b[i+1])].mean() if (m&(theta_own>=b[i])&(theta_own<b[i+1])).sum() else np.nan for i in range(16)]
ax[1].plot(np.degrees(bc), mp,'o-'); ax[1].set_xlabel('theta_own bin (deg)'); ax[1].set_ylabel('mean official phase'); ax[1].set_title('monotonic = good phase clock')
plt.tight_layout(); p=f'{OUT}/latent_phase_vs_rmm_phase.png'; plt.savefig(p,dpi=120,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 3 — Core estimators: first-harmonic, phase offset, regional, bootstrap

`A_f(x) = mean_t[ f(t,x) e^{i theta(t)} ]` over a day-mask. Regional phase uses the **complex sum** over
region longitudes (coherent), not an average of angles.

In [ ]:
REGIONS = {'IndianOcean':(60,90), 'MaritimeContinent':(100,130), 'WestPacific':(140,170)}
rmask = {k:(lons>=v[0])&(lons<=v[1]) for k,v in REGIONS.items()}

def harmonic(field, theta, mask):
    # complex first harmonic per longitude over masked days (NaN-robust) -> (nlon,)
    e = np.exp(1j*theta[mask])
    return np.nanmean(field[mask] * e[:,None], axis=0)

def wrapdeg(a):  # radians -> degrees in (-180,180]
    return np.degrees((a+np.pi)%(2*np.pi)-np.pi)

def region_offset(field, theta, mask, rm):
    Af = harmonic(field, theta, mask)[rm].sum()
    Ac = harmonic(conv,  theta, mask)[rm].sum()
    return wrapdeg(np.angle(Af)-np.angle(Ac)), np.abs(Af), np.abs(Ac)

def boot_offset(field, theta, mask, rm, B=500, seed=0):
    idx=np.where(mask)[0]; rng=np.random.default_rng(seed); out=[]
    e_all=np.exp(1j*theta)
    for _ in range(B):
        s=rng.choice(idx, size=len(idx), replace=True)
        Af=np.nanmean(field[s]*e_all[s,None],axis=0)[rm].sum()
        Ac=np.nanmean(conv[s] *e_all[s,None],axis=0)[rm].sum()
        out.append(wrapdeg(np.angle(Af)-np.angle(Ac)))
    out=np.array(out); c=np.angle(np.mean(np.exp(1j*np.radians(out))))
    d=wrapdeg(np.radians(out)-c)
    return float(np.degrees(c)), float(np.percentile(d,2.5)+np.degrees(c)), float(np.percentile(d,97.5)+np.degrees(c))

dqcol_dt = np.gradient(qcol, axis=0).astype(np.float32)   # NaN at the covered/uncovered boundary; nanmean handles it
print('estimators ready. regions:', REGIONS)

## Cell 4 — Phase-longitude composites (own-RMM), q_col / q_low / convection / u850

In [ ]:
NB=16
tb=np.linspace(-np.pi,np.pi,NB+1)
def comp_grid(field):
    g=np.full((NB,len(lons)),np.nan)
    for i in range(NB):
        mm=active&(theta_own>=tb[i])&(theta_own<tb[i+1])
        if mm.sum(): g[i]=field[mm].mean(0)
    return g
G={'q_col':comp_grid(qcol),'q_low':comp_grid(qlow),'conv (-OLR)':comp_grid(conv),'u850':comp_grid(u850)}
fig,ax=plt.subplots(1,4,figsize=(20,5),sharey=True)
for a,(k,gg) in zip(ax,G.items()):
    vmax=np.nanpercentile(np.abs(gg),95)
    im=a.pcolormesh(lons,np.degrees(.5*(tb[1:]+tb[:-1])),gg,cmap='RdBu_r',vmin=-vmax,vmax=vmax,shading='auto')
    a.set_title(k); a.set_xlabel('Longitude'); plt.colorbar(im,ax=a,fraction=.046)
ax[0].set_ylabel('latent phase theta_own (deg)')
fig.suptitle('Phase-longitude composites by own-RMM phase (active MJO days)',fontweight='bold')
plt.tight_layout(); p=f'{OUT}/phase_composites_q_olr.png'; plt.savefig(p,dpi=120,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 5 — delta_theta(field, convection) by longitude + regional table (q_col, q_low, dq/dt)

Reading: `q_col` near 0 = moisture-mode; `q_low` strongly positive = skeleton recharge; the **gap**
between them is the quantitative skeleton<->moisture-mode tension; `dq/dt` lead = propagation.

In [ ]:
FIELDS={'q_col':qcol,'q_low':qlow,'dq_col/dt':dqcol_dt}
colors={'q_col':'green','q_low':'orange','dq_col/dt':'purple'}
# by-longitude delta_theta
fig,ax=plt.subplots(figsize=(13,4.5))
for k,f in FIELDS.items():
    Af=harmonic(f,theta_own,active); Ac=harmonic(conv,theta_own,active)
    dphi=wrapdeg(np.angle(Af)-np.angle(Ac))
    w=np.abs(Af); w=w/w.max()
    ax.scatter(lons,dphi,s=8+30*w,c=colors[k],alpha=.5,label=k)
ax.axhline(0,color='k',lw=.6); ax.axhline(90,color='gray',ls=':'); ax.axhline(-90,color='gray',ls=':')
for k,v in REGIONS.items(): ax.axvspan(v[0],v[1],color='gray',alpha=.07)
ax.set_xlabel('Longitude'); ax.set_ylabel('delta_theta vs convection (deg)')
ax.set_title('Moisture/tendency phase lead over convection (size ~ harmonic amplitude)'); ax.legend()
plt.tight_layout(); p=f'{OUT}/delta_theta_by_longitude.png'; plt.savefig(p,dpi=120,bbox_inches='tight'); plt.show(); print('Saved',p)

# regional table with bootstrap CI
rows=[]
for k,f in FIELDS.items():
    for rk,rm in rmask.items():
        d,amp_f,amp_c=region_offset(f,theta_own,active,rm)
        c,lo,hi=boot_offset(f,theta_own,active,rm)
        rows.append({'field':k,'region':rk,'delta_theta_deg':round(d,1),
                     'ci_lo':round(lo,1),'ci_hi':round(hi,1),'amp_field':round(float(amp_f),3)})
reg_df=pd.DataFrame(rows); print(reg_df.to_string(index=False))
reg_df.to_csv(f'{OUT}/delta_theta_regions.csv',index=False)

fig,ax=plt.subplots(figsize=(11,4.5))
xs=np.arange(len(REGIONS)); off={'q_col':-.25,'q_low':0,'dq_col/dt':.25}
for k in FIELDS:
    sub=reg_df[reg_df.field==k]
    ax.errorbar(xs+off[k], sub['delta_theta_deg'],
                yerr=[sub['delta_theta_deg']-sub['ci_lo'], sub['ci_hi']-sub['delta_theta_deg']],
                fmt='o',capsize=4,color=colors[k],label=k)
ax.axhline(0,color='k',lw=.6,label='in-phase (moisture-mode)'); ax.axhline(-90,color='r',ls=':',label='quadrature -90 (skeleton)')
ax.set_xticks(xs); ax.set_xticklabels(list(REGIONS)); ax.set_ylabel('delta_theta (deg)')
ax.set_title('Moisture-convection phase offset by region (bootstrap 95% CI)'); ax.legend(fontsize=8)
plt.tight_layout(); p=f'{OUT}/delta_theta_regions_with_ci.png'; plt.savefig(p,dpi=120,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 6 — ENSO-stratified delta_theta (the headline)

In [ ]:
erows=[]
for cat in ['El Nino','Neutral','La Nina']:
    mm=active&(enso==cat)
    for k,f in FIELDS.items():
        for rk,rm in rmask.items():
            d,_,_=region_offset(f,theta_own,mm,rm)
            c,lo,hi=boot_offset(f,theta_own,mm,rm,B=400,seed=1)
            erows.append({'enso':cat,'field':k,'region':rk,'delta_theta_deg':round(d,1),
                          'ci_lo':round(lo,1),'ci_hi':round(hi,1),'n':int(mm.sum())})
enso_df=pd.DataFrame(erows); enso_df.to_csv(f'{OUT}/enso_stratified_delta_theta.csv',index=False)
print(enso_df.to_string(index=False))

fig,axes=plt.subplots(1,3,figsize=(17,4.5),sharey=True)
ecol={'El Nino':'#d62728','Neutral':'#7f7f7f','La Nina':'#1f77b4'}
for ax,k in zip(axes,FIELDS):
    for cat in ecol:
        sub=enso_df[(enso_df.field==k)&(enso_df.enso==cat)]
        xs=np.arange(len(REGIONS))+ (-.25 if cat=='El Nino' else (.25 if cat=='La Nina' else 0))
        ax.errorbar(xs, sub['delta_theta_deg'],
                    yerr=[sub['delta_theta_deg']-sub['ci_lo'], sub['ci_hi']-sub['delta_theta_deg']],
                    fmt='o',capsize=3,color=ecol[cat],label=cat)
    ax.axhline(0,color='k',lw=.6); ax.axhline(-90,color='r',ls=':')
    ax.set_xticks(np.arange(len(REGIONS))); ax.set_xticklabels(list(REGIONS),rotation=20,fontsize=8)
    ax.set_title(k)
axes[0].set_ylabel('delta_theta (deg)'); axes[0].legend(fontsize=8)
fig.suptitle('ENSO modulation of the moisture-convection phase relationship',fontweight='bold')
plt.tight_layout(); p=f'{OUT}/enso_stratified_delta_theta.png'; plt.savefig(p,dpi=120,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 7 — Rossby-Kelvin ratio (u850 asymmetry, §7 Fig 22) + ENSO

Proxy: per latent-phase composite of `u850'`, ratio of peak westerly (Rossby) to peak easterly (Kelvin).
Observed MJO ~1.0; Gill response ~2.2 (Wang & Lee 2017).

In [ ]:
def rk_ratio(mask):
    g=np.full((NB,len(lons)),np.nan)
    for i in range(NB):
        mm=mask&(theta_own>=tb[i])&(theta_own<tb[i+1])
        if mm.sum(): g[i]=u850[mm].mean(0)
    w=np.nanmax(g,axis=1); e=np.nanmax(-g,axis=1)              # peak westerly / peak easterly per phase
    r=w/np.maximum(e,1e-6)
    return float(np.nanmean(r)), r
rk_all,rk_phase=rk_ratio(active)
rk_enso={cat:rk_ratio(active&(enso==cat))[0] for cat in ['El Nino','Neutral','La Nina']}
print(f'Rossby-Kelvin ratio (all active): {rk_all:.2f}   (obs~1.0, Gill~2.2)')
print('by ENSO:', {k:round(v,2) for k,v in rk_enso.items()})

fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(np.degrees(.5*(tb[1:]+tb[:-1])), rk_phase,'o-'); ax[0].axhline(1,color='g',ls=':',label='obs~1')
ax[0].axhline(2.2,color='r',ls=':',label='Gill~2.2'); ax[0].set_xlabel('latent phase (deg)')
ax[0].set_ylabel('R-K ratio'); ax[0].set_title('R-K ratio by phase'); ax[0].legend(fontsize=8)
ax[1].bar(list(rk_enso),[rk_enso[k] for k in rk_enso],color=['#d62728','#7f7f7f','#1f77b4'])
ax[1].axhline(1,color='g',ls=':'); ax[1].axhline(2.2,color='r',ls=':'); ax[1].set_title('R-K ratio by ENSO')
plt.tight_layout(); p=f'{OUT}/rossby_kelvin_ratio.png'; plt.savefig(p,dpi=120,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 8 — BL-convergence lead (trio-interaction, §7)
convergence = -div_low; does it lead convection?

In [ ]:
if divl is None:
    print('div_low not available (uvplev not downloaded) -> skipping BL-convergence diagnostic.')
    bl_df = None
else:
    converg = -divl                                     # positive = convergence
    rows=[]
    for rk,rm in rmask.items():
        d,_,_=region_offset(converg,theta_own,active,rm)
        c,lo,hi=boot_offset(converg,theta_own,active,rm)
        rows.append({'region':rk,'delta_theta_conv_lead_deg':round(d,1),'ci_lo':round(lo,1),'ci_hi':round(hi,1)})
    bl_df=pd.DataFrame(rows); print(bl_df.to_string(index=False)); bl_df.to_csv(f'{OUT}/bl_convergence_lead.csv',index=False)
    Af=harmonic(converg,theta_own,active); Ac=harmonic(conv,theta_own,active)
    dphi=wrapdeg(np.angle(Af)-np.angle(Ac))
    fig,ax=plt.subplots(figsize=(13,4))
    ax.scatter(lons,dphi,s=8+30*np.abs(Af)/np.abs(Af).max(),c='teal',alpha=.6)
    ax.axhline(0,color='k',lw=.6); ax.axhline(90,color='gray',ls=':')
    for k,v in REGIONS.items(): ax.axvspan(v[0],v[1],color='gray',alpha=.07)
    ax.set_xlabel('Longitude'); ax.set_ylabel('BL convergence lead over convection (deg)')
    ax.set_title('Low-level convergence phase lead (positive = converg. leads convection, trio-interaction)')
    plt.tight_layout(); p=f'{OUT}/bl_convergence_lead.png'; plt.savefig(p,dpi=120,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 9 — Secondary / negative-control latents (SSL nb15, Barlow D=3)

Repeat the regional `delta_theta(q_col, conv)` using each learned latent's angle. Expectation:
SSL-temporal holds phase (coherent offsets), Barlow-Twins is phase-blind (incoherent / tiny amplitude).

In [ ]:
# Compare learned latents to own-RMM by DATE-ALIGNING them (they live on the bandpassed,
# edge-trimmed X_MJO_bp20_90 axis -> labels_aligned_mjo_bp20_90.csv), then measuring the same
# regional delta_theta(q_col, conv). Expectation: SSL-temporal recovers a coherent moisture lead;
# Barlow-Twins is phase-blind (incoherent / tiny harmonic amplitude).
from sklearn.decomposition import PCA as _PCA
full_row = {d:i for i,d in enumerate(pd.DatetimeIndex(labels['date']).normalize())}

def latent_offsets(name, emb_path, dates_csv):
    if not os.path.exists(emb_path):
        print(f'[skip] {name}: {emb_path} not found'); return []
    z = np.load(emb_path)
    if not os.path.exists(dates_csv):
        print(f'[skip] {name}: dates csv {dates_csv} not found'); return []
    bdates = pd.DatetimeIndex(pd.read_csv(dates_csv, parse_dates=['date'])['date']).normalize()
    if len(z) != len(bdates):
        print(f'[skip] {name}: emb rows {len(z)} != dates {len(bdates)}'); return []
    keep = np.array([d in full_row for d in bdates])
    fi = np.array([full_row[d] for d in bdates[keep]])          # full-axis row per embedding day
    zk = z[keep]
    zk2 = _PCA(2).fit_transform(zk - zk.mean(0)) if zk.shape[1] > 2 else (zk - zk.mean(0))
    th = np.arctan2(zk2[:,1], zk2[:,0])
    am = active[fi]                                             # active & moisture-covered on these days
    if am.sum() < 50:
        print(f'[skip] {name}: only {int(am.sum())} active&covered days'); return []
    if circ_corr(th[am], ph_ang[fi][am]) < 0: th = -th          # orient eastward
    e = np.exp(1j*th[am]); rows=[]
    for rk, rm in rmask.items():
        Af = np.nanmean(qcol[fi][am]*e[:,None], axis=0)[rm].sum()
        Ac = np.nanmean(conv[fi][am]*e[:,None], axis=0)[rm].sum()
        rows.append({'latent':name,'region':rk,'delta_theta_qcol':round(wrapdeg(np.angle(Af)-np.angle(Ac)),1),
                     'harm_amp':round(float(abs(Af)),3)})
    print(f'[ok] {name}: {len(zk)} emb rows, {int(am.sum())} active&covered, circ_corr(theta,phase)='
          f'{circ_corr(th[am], ph_ang[fi][am]):.2f}')
    return rows

srows=[{'latent':'own-RMM','region':rk,'delta_theta_qcol':round(region_offset(qcol,theta_own,active,rm)[0],1),
        'harm_amp':round(float(region_offset(qcol,theta_own,active,rm)[1]),3)} for rk,rm in rmask.items()]
BP = f'{PROC}/labels_aligned_mjo_bp20_90.csv'
srows += latent_offsets('SSL-temporal-2D', f'{MJO_DIR}/results/ssl/embeddings.npy', BP)
srows += latent_offsets('Barlow-D3',       f'{MJO_DIR}/barlow/D3/embeddings_z7.npy', BP)
srows += latent_offsets('Barlow-base',     f'{MJO_DIR}/barlow/embeddings_z7.npy',    BP)

sec_df=pd.DataFrame(srows)
print('\ndelta_theta(q_col) by latent and region (deg):')
print(sec_df.pivot(index='region',columns='latent',values='delta_theta_qcol'))
print('\nharmonic amplitude (low amp => phase-blind latent):')
print(sec_df.pivot(index='region',columns='latent',values='harm_amp'))
sec_df.to_csv(f'{OUT}/secondary_latent_delta_theta.csv',index=False)
print('Saved', f'{OUT}/secondary_latent_delta_theta.csv')

## Cell 10 — Summary JSON + interpretation

In [ ]:
summary={
 'phase_clock':'own-RMM (nb24), oriented to eastward', 'circ_corr_theta_vs_phase':round(cc,3),
 'n_active':int(active.sum()),
 'delta_theta_regions': reg_df.to_dict(orient='records'),
 'enso_stratified': enso_df.to_dict(orient='records'),
 'rossby_kelvin_ratio': {'all':round(rk_all,2), **{k:round(v,2) for k,v in rk_enso.items()}},
 'bl_convergence_lead': (bl_df.to_dict(orient='records') if bl_df is not None else 'skipped (no uvplev)'),
}
json.dump(summary, open(f'{OUT}/diagnostics_summary.json','w'), indent=2)
print('Saved', f'{OUT}/diagnostics_summary.json')
print('\n=== READ-OUT (active MJO days, own-RMM clock) ===')
for rk in REGIONS:
    qc=reg_df[(reg_df.field=="q_col")&(reg_df.region==rk)]['delta_theta_deg'].values[0]
    ql=reg_df[(reg_df.field=="q_low")&(reg_df.region==rk)]['delta_theta_deg'].values[0]
    print(f'{rk:18s}  d(q_col)={qc:+5.0f} deg   d(q_low)={ql:+5.0f} deg   gap={ql-qc:+5.0f} deg')
print(f'Rossby-Kelvin ratio = {rk_all:.2f} (obs~1.0, Gill~2.2)')
print('Interpretation: negative = field LEADS convection (east). q_col small lead ~in-phase')
print('  -> moisture-mode; q_low larger lead toward -90 -> skeleton recharge;')
print('the q_low-q_col gap and ENSO-dependence are the discriminating, novel results.')

---
## Done!
Figures + tables in `MJO/moisture_constraints/results/diagnostics/`. Send back
`diagnostics_summary.json` + the PNGs. The **decision gate** (per the experiment plan): the measured
`delta_theta` and its ENSO dependence determine which (if any) auxiliary/physics constraint to train next.

---
*DDCS Project | jh9141@nyu.edu*